# Mouse brain — CellSTIC tutorial
**Dataset:** mouse brain 5M 20 µm (RNA, ADT, ATAC, H3K27ac, H3K27me3). See `data/mouse_brain/README.md`.


## 1. Repository path


In [ ]:
from pathlib import Path
import sys
import torch

# Add project root to path
try:
    project_root = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    project_root = _cwd if (_cwd / "model").is_dir() else _cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root.resolve()}")

## 2. Imports


In [ ]:
from utils.tools.seed_utils import set_global_seed
# Local import to avoid hard dependency at module import time
from utils.data.peak_count import run_peak_count
from utils.loader import load_mouse_brain, load_mouse_brain_gene_scores
from utils.train import load_config
from model.cellstic import CellSTIC
from pipeline.trainer import CellSTICTrainer
from pipeline.evaluator import CellSTICEvaluator
from pipeline.analyzer import SingleLevelAnalysis
from utils import ModelUtils
from utils.metrics import MetricsComputer, SpatialVisualizer, UMAPVisualizer, get_custom_palette

set_global_seed()  # once before train/eval; pipeline does not set RNG here


## 3. Configuration


In [ ]:
# --- Run configuration ---
WORK_DIR = str(project_root / "data" / "mouse_brain")
RUN_OPTIONAL_MODALITY_VIS = True  # per-modality region UMAP (feat / gene_score)
RUN_OPTIONAL_CELLTYPE_VIS = True  # RNA cell-type spatial map + UMAP
RUN_TRAIN = True
RUN_PEAK_COUNT = False  # if True, run ArchR/MACS2 fragment pipeline (slow; requires fragment .tsv.gz under raw_path)
MACS2_PATH = "/home/wangshuai/anaconda3/envs/spaGem/bin/macs2"
R_HOME = "/home/wangshuai/anaconda3/envs/R4.3.1-3/"
PEAK_COUNT_THREADS = 8
PEAK_COUNT_GENOME = "mm10"

work_dir = Path(WORK_DIR)
output_path = work_dir / "output"
raw_path = work_dir / "raw"
model_path = work_dir / "model"
preprocess_path = work_dir / "preprocess"
config_path = work_dir / "config" / "mouse_brain_config.yaml"

for _p in [output_path, raw_path, model_path, preprocess_path]:
    _p.mkdir(parents=True, exist_ok=True)

config = load_config(config_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Work dir: {work_dir}")
print(f"Raw path: {raw_path}")
print(f"Preprocess path: {preprocess_path}")
print(f"Model path: {model_path}")
print(f"Output path: {output_path}")
print(f"Config path: {config_path}")
print(f"Device: {device}")
print(f"RUN_OPTIONAL_MODALITY_VIS = {RUN_OPTIONAL_MODALITY_VIS}")
print(f"RUN_OPTIONAL_CELLTYPE_VIS  = {RUN_OPTIONAL_CELLTYPE_VIS}")
print(f"RUN_PEAK_COUNT             = {RUN_PEAK_COUNT}")


## 4. Peak count (ATAC / histone fragments)

Optional **offline-style** step: build gene-score / peak matrices from `fragments.tsv.gz` via `utils.data.peak_count.run_peak_count` (ArchR + MACS2). Outputs go under `preprocess_path/<modality>/`. Turn on with `RUN_PEAK_COUNT = True` in the configuration cell and **edit fragment filenames** in the code below to match your `raw/` layout.


In [ ]:
if RUN_PEAK_COUNT:
    # Fragment filenames relative to raw_path / <modality> / (same layout as LRHT-CCC mouse_brain runner)
    modality_configs = [
        ("atac", "GSM8494157_5M_20um_ATAC.fragments.tsv.gz"),
        ("h3k27ac", "GSM8494157_5M_20um_H3K27ac.fragments.tsv.gz"),
        ("h3k27me3", "GSM8494157_5M_20um_H3K27me3.fragments.tsv.gz"),
    ]
    for modality, fname in modality_configs:
        fragments_path = raw_path / modality / fname
        output_dir = preprocess_path / modality
        print(f"[peak_count] {modality}: {fragments_path} -> {output_dir}")
        run_peak_count(
            fragments_file=str(fragments_path),
            output_dir=str(output_dir),
            sample_name=None,
            genome=PEAK_COUNT_GENOME,
            threads=PEAK_COUNT_THREADS,
            tile_size=5000,
            add_tile_mat=False,
            add_gene_score_mat=True,
            add_peak_mat=True,
            macs2_path=MACS2_PATH,
            r_home=R_HOME,
            rscript_path=None,
            convert_to_csv=True,
        )
else:
    print("Skipped peak_count (set RUN_PEAK_COUNT = True after editing fragment paths).")


## 5. Optional modality visualization

Per-modality **region UMAP** exports using `adata.obsm['feat']` (RNA/ADT) and `adata.obsm['gene_score']` (ATAC, H3K27ac, H3K27me3). Does **not** require a trained `CellSTIC` model. Prepare gene-score matrices offline (e.g. `run_peak_count(...)`) if chromatin UMAPs are missing.


In [ ]:
if RUN_OPTIONAL_MODALITY_VIS:
    (
        rna_adata_umap,
        adt_adata_umap,
        atac_adata_umap,
        h3k27ac_adata_umap,
        h3k27me3_adata_umap,
        _,
    ) = load_mouse_brain(raw_path, preprocess_path, use_cache=True)

    # RNA / ADT use precomputed low-dimensional features
    modality_map_rna_adt = {
        "RNA": rna_adata_umap,
        "ADT": adt_adata_umap,
    }
    # Region UMAP + clustering metrics for RNA/ADT using obsm['feat']
    for modality_name, adata in modality_map_rna_adt.items():
        if "feat" not in adata.obsm:
            raise ValueError(f"{modality_name} missing adata.obsm['feat'], cannot run per-modality UMAP.")
        MetricsComputer.run_region_umap_metrics_export(
            adata=adata,
            save_dir=output_path / modality_name.lower(),
            feature_key="feat",
            true_labels=None,
            cluster_key=None,
            n_clusters=9,
        )

    # ATAC / H3K27ac / H3K27me3 use gene score matrices
    atac_gs, h3k27ac_gs, h3k27me3_gs = load_mouse_brain_gene_scores(raw_path, preprocess_path)
    modality_map_gs = {
        "ATAC": atac_gs,
        "H3K27ac": h3k27ac_gs,
        "H3K27me3": h3k27me3_gs,
    }
    # Region UMAP for chromatin modalities using peak-derived gene-score embeddings
    for modality_name, adata in modality_map_gs.items():
        if "gene_score" not in adata.obsm:
            raise ValueError(f"{modality_name} missing adata.obsm['gene_score'], cannot run gene-score UMAP.")
        MetricsComputer.run_region_umap_metrics_export(
            adata=adata,
            save_dir=output_path / f"{modality_name.lower()}_gene_score",
            feature_key="gene_score",
            true_labels=None,
            cluster_key=None,
            n_clusters=9,
        )
else:
    print("Skipped optional modality UMAP exports.")


## 6. Optional cell-type visualization

RNA **spatial** layout and **UMAP** from `obs['cell_type']` and `obsm['feat']`. Does **not** require a trained model.


In [ ]:
if RUN_OPTIONAL_CELLTYPE_VIS:
    (
        rna_adata_ct,
        _,
        _,
        _,
        _,
        _,
    ) = load_mouse_brain(raw_path, preprocess_path, use_cache=True)

    if "cell_type" not in rna_adata_ct.obs:
        raise ValueError("`cell_type` not found in rna_adata.obs; cannot visualize cell-type spatial distribution.")

    rna_adata_ct = rna_adata_ct.copy()
    rna_adata_ct.obs["cluster"] = rna_adata_ct.obs["cell_type"].astype(str).astype("category")

    save_path_ct = output_path / "cell_type_distribution" / "spatial_domain.svg"
    save_path_ct.parent.mkdir(parents=True, exist_ok=True)
    palette = get_custom_palette(len(rna_adata_ct.obs["cluster"].cat.categories))
    # Spatial map: RNA cell-type labels on tissue coordinates
    SpatialVisualizer.generate_spatial_domain_visualization(
        adata_source=rna_adata_ct,
        save_path=str(save_path_ct),
        palette=palette,
    )
    print(f"Cell-type spatial distribution saved to {save_path_ct}")

    if "feat" not in rna_adata_ct.obsm:
        raise ValueError("`feat` not found in rna_adata.obsm; cannot generate UMAP for cell types.")

    umap_path_ct = output_path / "cell_type_distribution" / "umap.svg"
    # UMAP of RNA features colored by cell-type clusters
    UMAPVisualizer.generate_visualization(
        node_features=torch.tensor(rna_adata_ct.obsm["feat"], dtype=torch.float32),
        adata_source=rna_adata_ct,
        save_path=str(umap_path_ct),
        palette=palette,
        add_kde_contour=False,
        add_cluster_labels=False,
        n_neighbors=40,
        color_key="cluster",
    )
    print(f"Cell-type UMAP saved to {umap_path_ct}")
else:
    print("Skipped optional cell-type visualization.")


## 7. Load multimodal data

Load RNA, ADT, ATAC, and histone modalities plus the ligand–receptor map. Run this before training or evaluation.


In [ ]:
(
    rna_adata,
    adt_adata,
    atac_adata,
    h3k27ac_adata,
    h3k27me3_adata,
    ligand_receptor_map,
) = load_mouse_brain(raw_path, preprocess_path, use_cache=True)

modality_datas = [rna_adata, adt_adata, atac_adata, h3k27ac_adata, h3k27me3_adata]


## 8. Train or load model

Set `RUN_TRAIN = True` in the configuration cell to train from scratch; otherwise the saved checkpoint under `model_path` is loaded.


In [ ]:
if RUN_TRAIN:
    model = CellSTIC(config.model, device)
    CellSTICTrainer(
        model,
        config,
        model_path=model_path,
        device=device,
        ligand_receptor_map=ligand_receptor_map,
    ).train(
        modality_datas=modality_datas,
        is_train_ccc=True,
        is_train_feature=True,
    )
else:
    model = ModelUtils.load_model(model_path=model_path, config=config, device=device)


## 9. Evaluate features and CCC

Clustering / reconstruction metrics on fused modalities, then ligand–receptor edge probabilities (`evaluate_ccc_precision`).


In [ ]:
# Wrap model + paths for downstream evaluation APIs
evaluator = CellSTICEvaluator(
    model,
    config,
    ligand_receptor_map=ligand_receptor_map,
    model_path=model_path,
    output_path=output_path,
    device=device,
)
# Fused-feature clustering / reconstruction metrics (multi-modality)
evaluator.evaluate_mutiple_feature(
    modality_datas=modality_datas,
    auto_n_clusters=9,
    true_labels=None,
)
# Single-level CCC edge probabilities
pos_edge_probs_np, edge_type_map, m0 = evaluator.evaluate_ccc_precision(
    modality_datas=modality_datas,
)


## 10. Single-level analysis

Spatial LR maps, heatmaps, and sender/receiver summaries on the CCC evaluation output.


In [ ]:
# Single-level analyzer on CCC outputs (RNA slice `m0`, curated LR filter)
single_level_analyzer = SingleLevelAnalysis(
    adata=m0,
    output_path=output_path,
    pos_edge_probs_np=pos_edge_probs_np,
    edge_type_map=edge_type_map,
    cell_type_key="cell_type",
    threshold=0.5,
    lr_filter=["Col6a3:Sdc4", "Penk:Oprk1", "Cd200:Cd200r4"],
)
# Spatial LR pair maps (selected ligand–receptor list)
single_level_analyzer.run_lr_spatial()
# Compact LR heatmaps for the filtered edge set
single_level_analyzer.run_simple_heatmaps(font_size=12)
# Sender/receiver strength stacked bar summary
single_level_analyzer.run_stacked_bar(figsize=(1, 5))
